# Phase d'avant modelisation

**Objectif de cette partie :** préparer les données, créer de nouvelles variables pertinentes (feature engineering) et construire le pipeline de prétraitement, **en évitant toute fuite de données**.

Ce notebook est la première étape avant la modélisation (Notebook 3).

**Plan :**
1. Lexique express pour débutants
2. Chargement du dataset nettoyé (avec valeurs manquantes volontairement conservées)
3. Séparation train/test (**avant** toute autre étape dépendant des données)
4. Feature engineering (sans fuite)
5. Encodage par risque avec lissage (sans fuite)
6. Préparation du pipeline (imputation + encodage + normalisation)
7. Sauvegarde des objets pour le Notebook 3

## 🧭 Concepts clés avant de commencer

- **Apprentissage supervisé** : on montre au modèle des exemples passés (commerçants avec leur vrai statut de défaut) pour qu'il apprenne à prédire le statut de nouveaux commerçants.
- **Pipeline** : un enchaînement d'étapes de transformation des données (imputation, encodage, normalisation...) suivi d'un modèle, le tout traité comme un seul bloc. Avantage : on ne peut pas "oublier" une étape ou l'appliquer au mauvais moment.
- **Fuite de données (leakage)** : utiliser, même involontairement, une information du jeu de test dans l'entraînement, ce qui fausse l'évaluation du modèle.

**Pré-requis technique (à installer une seule fois, si besoin) :**
```
pip install pandas numpy matplotlib seaborn scikit-learn xgboost lightgbm catboost
```

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)

## 1. Chargement du dataset nettoyé

⚠️ Ce fichier contient **encore des valeurs manquantes** — c'est volontaire (voir notebook 1). On va les traiter correctement ci-dessous, **après** avoir séparé train et test.

In [2]:
df = pd.read_csv('sencredit_clean.csv')
print(f"Dimensions : {df.shape}")
print("\nValeurs manquantes par colonne :")
display(df.isnull().sum())

X = df.drop(['Defaut_Paiement', 'ID_Commercant'], axis=1)
y = df['Defaut_Paiement']

Dimensions : (10000, 8)

Valeurs manquantes par colonne :


,0
ID_Commercant,0
Zone_Dakar,0
Secteur_Activite,517
Volume_Mensuel_FCFA,0
Evolution_Ventes_3M_Pct,600
Nb_Transferts_Recus_Mois,0
Anciennete_Mois,392
Defaut_Paiement,0


## 2. Séparation train / test — **avant tout le reste**

**Analogie :** pensez à un·e élève qui prépare un examen. Le "train" est le cahier d'exercices sur lequel il/elle s'entraîne. Le "test" est l'examen final, qu'il/elle ne doit voir qu'une seule fois, à la toute fin, dans des conditions réalistes.

**Règle d'or appliquée dans ce notebook :** toute étape qui "apprend" quelque chose à partir des données (une moyenne, une médiane, un taux de risque par zone...) doit être calculée **uniquement sur le train**, puis appliquée telle quelle au test. C'est pour cela qu'on sépare train/test **en tout premier**, avant même de créer de nouvelles variables.

Le paramètre `stratify=y` garantit que la proportion de bons/mauvais payeurs est la même dans le train et dans le test — c'est important vu le déséquilibre de classes observé dans le notebook 1.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Taille du train : {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Taille du test  : {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")
print(f"\nTaux de defaut dans le train : {y_train.mean()*100:.2f}%")
print(f"Taux de defaut dans le test  : {y_test.mean()*100:.2f}%")

# Copie de travail avec la cible, utile pour les agregations par groupe (etape 4)
X_train_with_target = X_train.copy()
X_train_with_target['Defaut_Paiement'] = y_train.values

Taille du train : 7000 (70.0%)
Taille du test  : 3000 (30.0%)

Taux de defaut dans le train : 20.00%
Taux de defaut dans le test  : 20.00%


## 3. Feature engineering (création de nouvelles variables)

On va créer de nouvelles variables ("features") à partir des colonnes existantes, dans l'espoir qu'elles aident le modèle à mieux distinguer les bons des mauvais payeurs.

**Deux types de features, deux traitements différents :**

1. **Features "ligne par ligne"** (ratios, interactions) : elles ne dépendent que des valeurs de la ligne elle-même (ex : volume ÷ ancienneté). Elles peuvent être calculées indépendamment sur le train et sur le test, sans aucun risque de fuite.
2. **Features "par groupe"** (ex : taux de risque moyen par zone) : elles dépendent d'une statistique calculée sur *plusieurs* lignes, potentiellement liée à la cible. Celles-ci **doivent** être calculées uniquement sur le train (voir §4, plus loin).

On commence par les features "ligne par ligne".

In [4]:
for jeu in [X_train, X_test]:
    jeu['Vol_par_mois'] = jeu['Volume_Mensuel_FCFA'] / (jeu['Anciennete_Mois'] + 1)
    jeu['Transferts_par_mois'] = jeu['Nb_Transferts_Recus_Mois'] / (jeu['Anciennete_Mois'] + 1)
    jeu['Vol_evo_interaction'] = jeu['Volume_Mensuel_FCFA'] * (jeu['Evolution_Ventes_3M_Pct'] / 100)

    # Tranches d'anciennete : des bornes fixes, definies a l'avance, donc pas de fuite possible
    jeu['Anciennete_cat'] = pd.cut(
        jeu['Anciennete_Mois'],
        bins=[0, 6, 18, 36, 120],
        labels=['Debutant', 'Intermediaire', 'Etabli', 'Senior'],
        right=False
    )

**Pourquoi le `+ 1` dans les ratios ?** Pour éviter une division par zéro si `Anciennete_Mois` vaut 0 (nouveau commerçant). C'est une astuce courante appelée "lissage additif" simple.

**Pourquoi ces variables peuvent contenir des `NaN` ?** Si `Volume_Mensuel_FCFA` ou `Anciennete_Mois` est manquant (`NaN`) sur une ligne, alors `Vol_par_mois` sera aussi `NaN` pour cette ligne (un calcul avec une valeur manquante donne un résultat manquant). C'est normal — l'imputation finale se fera dans le pipeline (§6), qui comblera aussi ces nouvelles colonnes.

## 4. Encodage du risque par zone et par secteur (avec lissage, sans fuite)

### L'idée

On aimerait résumer, pour chaque zone géographique et chaque secteur d'activité, "à quel point les commerçants de ce groupe font-ils souvent défaut, historiquement ?" C'est ce qu'on appelle un **encodage par cible (target encoding)** : on remplace une catégorie (ex : "Sandaga") par une statistique liée à la cible (ex : le taux de défaut moyen des commerçants de Sandaga).

### Le risque de fuite

Si on calcule ce taux moyen en utilisant **toutes** les données (train + test), alors la valeur utilisée pour une ligne du test contient... une information sur son propre statut de défaut, potentiellement mélangée dans la moyenne du groupe ! C'est une fuite subtile mais réelle.

**Solution :** on calcule ces moyennes **uniquement sur le train**, puis on les applique telles quelles au test (une zone du test recevra le taux de risque "appris" sur le train, jamais recalculé avec les vraies valeurs du test).

### Le problème des petits groupes : pourquoi lisser ?

Imaginons qu'une zone n'ait que 2 commerçants dans le train, et que les deux soient en défaut. Le taux "brut" serait de 100% — un chiffre extrême et peu fiable, basé sur seulement 2 observations. Le modèle risquerait de sur-interpréter ce chiffre (**surapprentissage / overfitting**).

**Solution : le lissage (smoothing).** On mélange le taux du groupe avec le taux moyen global, en donnant plus de poids au taux global quand le groupe est petit, et plus de poids au taux du groupe quand il est grand (beaucoup d'observations = statistique plus fiable). La formule utilisée est :

```
taux_lisse = (n_groupe x taux_groupe + m x taux_global) / (n_groupe + m)
```

où `m` est un paramètre qui contrôle la force du lissage (plus `m` est grand, plus on "tire" vers la moyenne globale pour les petits groupes).

In [5]:
def encoder_risque_lisse(train_avec_cible, colonne, cible, m=10):
    '''
    Calcule un encodage de risque lisse pour une colonne categorielle,
    a partir du train UNIQUEMENT.

    Retourne :
    - un dictionnaire {categorie: taux_lisse} a appliquer au train et au test
    - le taux global (utilise pour les categories jamais vues dans le train)
    '''
    taux_global = train_avec_cible[cible].mean()
    stats = train_avec_cible.groupby(colonne)[cible].agg(['mean', 'count'])
    taux_lisse = (stats['count'] * stats['mean'] + m * taux_global) / (stats['count'] + m)
    return taux_lisse.to_dict(), taux_global


# Calcul sur le train uniquement
mapping_zone, taux_global_zone = encoder_risque_lisse(X_train_with_target, 'Zone_Dakar', 'Defaut_Paiement', m=10)
mapping_secteur, taux_global_secteur = encoder_risque_lisse(X_train_with_target, 'Secteur_Activite', 'Defaut_Paiement', m=10)

# Application au train ET au test avec le MEME mapping (appris sur le train)
for jeu in [X_train, X_test]:
    jeu['risque_zone'] = jeu['Zone_Dakar'].map(mapping_zone).fillna(taux_global_zone)
    jeu['risque_secteur'] = jeu['Secteur_Activite'].map(mapping_secteur).fillna(taux_global_secteur)

print(f"Taux de defaut global (train) : {taux_global_zone:.4f}")
print("\nExemple de taux lisses par zone (train) :")
display(pd.Series(mapping_zone).sort_values(ascending=False).head())

Taux de defaut global (train) : 0.2000

Exemple de taux lisses par zone (train) :


,0
Parcelles,0.209841
Medina,0.202413
Yoff,0.200542
Guediawaye,0.200299
Pikine,0.199519


**Pourquoi `.fillna(taux_global_...)` après le `.map()` ?** Si une zone ou un secteur présent dans le test n'existait pas du tout dans le train (cas rare mais possible), le `.map()` renverrait `NaN` pour cette ligne. On la remplace alors par le taux global, la meilleure estimation "par défaut" disponible.

**Ne pas oublier :** on retire mentalement la colonne cible de `X_train` — elle n'a été ajoutée que temporairement dans `X_train_with_target`, juste pour ce calcul. Vérifions-le explicitement :

In [6]:
# X_train ne doit jamais contenir la colonne cible
assert 'Defaut_Paiement' not in X_train.columns, "La cible ne doit pas etre dans X_train !"
print("Verification OK : la cible n'est pas dans les features.")

Verification OK : la cible n'est pas dans les features.


## 5. Pourquoi on n'encode PAS `Zone_Dakar` et `Secteur_Activite` une deuxième fois


C'est redondant : les deux représentations portent essentiellement la même information (une catégorie / un résumé du risque de cette catégorie), ce qui peut :
- rendre le modèle inutilement complexe ;
- compliquer l'interprétation ("est-ce que c'est la zone brute ou le taux de risque de la zone qui compte ?") ;
- créer de la colinéarité (deux variables très liées entre elles), ce qui perturbe surtout les modèles linéaires comme la régression logistique.

**Décision :** on garde uniquement `risque_zone` et `risque_secteur` (numériques, informatives, moins nombreuses), et on ne référence **pas** `Zone_Dakar` / `Secteur_Activite` brutes dans le préprocesseur (§6). La variable `Anciennete_cat`, elle, n'est **pas** dérivée de la cible (ce sont des tranches fixes) : elle est donc conservée en one-hot sans souci.

*Astuce technique :* comme le préprocesseur ne référence explicitement que les colonnes utiles (via les listes `num_features` et `cat_features` ci-dessous), les colonnes brutes `Zone_Dakar` et `Secteur_Activite` restent présentes dans `X_train`/`X_test` mais sont automatiquement ignorées par le pipeline — pas besoin de les supprimer manuellement.

## 6. Préparation du pipeline de prétraitement

Un **pipeline** enchaîne plusieurs étapes de transformation, appliquées dans l'ordre, de façon identique au train et au test. C'est essentiel pour éviter les erreurs et les fuites : chaque étape "apprend" ses paramètres (comme la médiane pour l'imputation) **uniquement quand on l'entraîne sur le train** (`fit`), puis les réapplique tels quels au test (`transform`), sans jamais réapprendre sur le test.

**Étapes pour les variables numériques :**
1. `SimpleImputer(strategy='median')` : remplace les `NaN` par la médiane — calculée **uniquement sur le train** grâce au pipeline. On utilise la médiane pour toutes les colonnes numériques par souci de simplicité et de cohérence (elle est robuste aux valeurs extrêmes).
2. `StandardScaler()` : met toutes les variables numériques à la même échelle (moyenne 0, écart-type 1). Utile car certains modèles (comme la régression logistique) sont sensibles à l'échelle des variables (un volume en millions de FCFA et un pourcentage entre -50 et +50 n'ont pas la même échelle naturellement).

**Étapes pour les variables catégorielles :**
1. `SimpleImputer(strategy='most_frequent')` : remplace les `NaN` par la catégorie la plus fréquente — calculée sur le train.
2. `OneHotEncoder()` : transforme chaque catégorie en colonnes 0/1 (ex : `Anciennete_cat_Etabli`, `Anciennete_cat_Senior`, ...).

In [7]:
num_features = ['Volume_Mensuel_FCFA', 'Evolution_Ventes_3M_Pct', 'Nb_Transferts_Recus_Mois',
                'Anciennete_Mois', 'Vol_par_mois', 'Transferts_par_mois',
                'Vol_evo_interaction', 'risque_zone', 'risque_secteur']
cat_features = ['Anciennete_cat']  # Zone_Dakar et Secteur_Activite volontairement exclues (voir etape 5)

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
    ]), cat_features)
])

## 7. Sauvegarde pour le Notebook 3

Les variables `X_train`, `X_test`, `y_train`, `y_test` et `preprocessor` sont prêtes. Nous allons utiliser `joblib` pour les transmettre au prochain notebook sans avoir à recalculer tout le feature engineering.

In [8]:
import joblib

# Regroupement de vos variables dans un dictionnaire
data_to_save = {
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train,
    "y_test": y_test,
    "preprocessor": preprocessor,
    "num_features": num_features,
    "cat_features": cat_features,
}

# Sauvegarde locale dans le dossier de session
joblib.dump(data_to_save, "sencredit_preprocessed.pkl")
print("Fichier 'sencredit_preprocessed.pkl' créé avec succès !")

Fichier 'sencredit_preprocessed.pkl' créé avec succès !
